## WALKING-CLIFF Sarsa

### Alberto Bono 3962994

Importing required modules

In [ ]:
from abc import ABC, abstractmethod
import random
import sys
import gym

Building the environment model (simulator)

In [ ]:
class BaseEnvironment(ABC):
    
    @abstractmethod
    # Override this method to return the set of states
    def get_states(self):
        pass
    
    @abstractmethod
    # Override this method to return the set of actions available in the state
    def get_actions(self, state):
        pass
    
    @abstractmethod
    def get_all_actions(self):
        pass
    
    @abstractmethod
    # Ovveride this method to implement action execution
    def do_action_and_get_reward(self, action):
        pass

Action Space: The actions available to the agent in each state are defined as follows:

- UP: Move one step upwards.
- DOWN: Move one step downwards.
- LEFT: Move one step to the left.
- RIGHT: Move one step to the right.
- STAY: Stay in the current position.

Observation Space: The agent's observation space is defined by the grid of the environment. Each cell in the grid represents a state, and the agent's position within the grid is its current observation.

Episode Start: Each episode starts with the agent positioned at the bottom-left corner of the grid, i.e., (3, 0).

Rewards:

- Moving into a cliff cell (bottom row except for the starting and ending points) results in a reward of -100.
- Moving into any other cell incurs a reward of -1.
- Reaching the goal cell (bottom-right corner) yields a reward of +10.

Episode End: An episode ends when the agent reaches the goal cell (3, 11) or steps into a cliff cell, resulting in a reward of -100.

In [ ]:
class WalkingCliff(BaseEnvironment):
    # Define actions as tuples representing movement directions
    UP = (-1,0)
    DOWN = (1,0)
    LEFT = (0,-1)
    RIGHT = (0,1)
    STAY = (0,0)
    
    # Define labels for actions
    label = {(-1,0):"UP", (1,0):"DOWN", (0,-1):"LEFT", (0,1):"RIGHT", (0,0):"STAY"}
    
    def __init__(self):
         # Initialize the environment with states and actions
        self.states = list()
        for ri in range(4):
            for co in range(12):
                self.states.append((ri,co))
                
        # Define actions for each state
        self.actions = dict()
        self.actions[(0,0)] = [WalkingCliff.RIGHT, WalkingCliff.DOWN]
        self.actions[(0,11)] = [WalkingCliff.LEFT, WalkingCliff.DOWN]
        for i in range(1,11): 
            self.actions[(0,i)] = [WalkingCliff.RIGHT, WalkingCliff.LEFT, WalkingCliff.DOWN]
            self.actions[(1,i)] = [WalkingCliff.RIGHT, WalkingCliff.LEFT, WalkingCliff.DOWN,WalkingCliff.UP]
            self.actions[(2,i)] = [WalkingCliff.RIGHT, WalkingCliff.LEFT, WalkingCliff.DOWN,WalkingCliff.UP]
            #cliff
            self.actions[(3,i)] = [WalkingCliff.DOWN,WalkingCliff.STAY, WalkingCliff.LEFT, WalkingCliff.UP]
            
        for j in range(1,3):
            self.actions[(j,0)] = [WalkingCliff.RIGHT, WalkingCliff.UP, WalkingCliff.DOWN]
            self.actions[(j,11)] = [WalkingCliff.LEFT, WalkingCliff.DOWN,WalkingCliff.UP]
        self.actions[(3,0)] = [WalkingCliff.RIGHT, WalkingCliff.UP]
        self.actions[(3,11)] = [WalkingCliff.STAY, WalkingCliff.LEFT, WalkingCliff.UP]
        # Define all possible actions
        self.all_actions = [WalkingCliff.STAY, WalkingCliff.RIGHT, WalkingCliff.LEFT, WalkingCliff.DOWN, WalkingCliff.UP]
        # Initialize the current state
        self.current_state = (3,0)
    
    def get_states(self):
        return self.states
        
    def get_actions(self, state):
        return self.actions[state]
    
    def get_all_actions(self, state):
        return self.all_actions
    
    def transition(self, current_state, action):
        return (current_state[0] + action[0], current_state[1] + action[1]) 
    
    def do_action_and_get_reward(self, action):
        # If the action is not executable in the state, do nothing and return reward 0
        if action not in self.get_actions(self.current_state):
            return 0
        # Compute the next state based on the action
        self.current_state = self.transition(self.current_state, action)
        
        # If the agent moves into a cliff cell, reset and return reward -100
        if self.current_state[0]== 3 and 1<= self.current_state[1]<=10:
            self.reset()
            return -100
        # If the agent reaches the goal cell, return reward 10
        elif (self.current_state == (3,11)):
            return 10
        # For all other states, return reward -1
        else:
            return -1
        
    def task_done(self):
        # Check if the task is done (agent reached the goal state)
        return self.current_state == (3, 11)
    
    def reset(self):
        # Reset the environment (agent position to starting point)
        self.current_state = (3, 0)

## Computing policies

Base class for all policies. The method apply enforces the policy, the method update changes the policy.

In [ ]:
#SARSA and Q_Learning do not update
#the policy straight away( no update method)
class BasePolicy(ABC):
    
    @abstractmethod
    # Ovveride this method to implement policy application
    # Returns the action given the state
    def apply(self, state):
        pass

A concrete policy to initialize the search for an optimal policy. Implements an  ??
 -greedy technique for action choice.


The Epsilon-Greedy policy is a common strategy used in reinforcement learning to balance exploration and exploitation. It allows the agent to explore new actions (exploration) while also exploiting its current knowledge to choose actions that are estimated to be the best (exploitation).

Epsilon Parameter: The Epsilon-Greedy policy addresses the exploration-exploitation trade-off by introducing an epsilon parameter (?). This parameter determines the balance between exploration and exploitation. Epsilon is a value between 0 and 1.

With probability ? (epsilon), the agent chooses to explore by selecting a random action from the set of available actions. This encourages the agent to try out new actions and gather more information about the environment.
With probability 1 - ?, the agent chooses to exploit its current knowledge by selecting the action with the highest estimated value obtained from Q-table

When ? is high (close to 1), the agent is more likely to explore and select random actions. This can be beneficial during the early stages of learning when the agent's knowledge about the environment is limited.
When ? is low (close to 0), the agent is more likely to exploit and select actions based on its current knowledge. This is advantageous when the agent has learned a good approximation of the optimal policy and wants to maximize rewards based on its existing knowledge.

As the agent interacts with the environment and receives reward  it updates the Q-table based on the observed outcomes. Over time, the agent learns to strike a balance between exploration and exploitation to maximize its cumulative rewards.

In [ ]:
class EpsilonGreedyPolicy(BasePolicy):
    #We need to have access to the simulator and the Q_Table
    def __init__(self, environment, Q_table, epsilon):
        self.environment = environment
        self.Q_table = Q_table
        self.epsilon = epsilon
    
    # Apply the policy to select an action given the current state
    def apply(self, state):
        #Get all the action available in the state
        actions = self.environment.get_actions(state)
        #Let's extract a random number between 0 and 1 
        #(Hp. 0 epsilon < 1, so prob. of extracting a number less than epsilon is epsilon)
        if random.random() < self.epsilon:
            # Choose an action at random with probability epsilon
            return random.choice(actions)
        else:
            # Choose the best action accordin to Q_table with probability 1-epsilon
            # If all actions have the same Q-value then break ties randomly
            max_action_value = -1* sys.float_info.max
            best_action = random.choice(actions)
            for action in actions:
                if self.Q_table[state][action] > max_action_value:
                    max_action_value = self.Q_table[state][action]
                    best_action = action
            return action

## SARSA algorithm (on-policy reinforcement Learning)

SARSA (State-Action-Reward-State-Action) is an on-policy temporal difference reinforcement learning algorithm. It learns the Q-values (action values) for each state-action pair based on the observed transitions and rewards. SARSA updates the Q-values using the observed transitions in a sample episode.



In [ ]:
class SARSA:
    # Initialize the SARSA algorithm with the environment
    # gamma := discount factor for discounted sum of rewards
    # alpha := step size parameter for appproximate solution of Bellman eq.
    # epsilon := the parameter for epsilom greedy policy
    # episodes := number of episodes to try
    def __init__(self, environment, gamma, alpha, epsilon, episodes):
        self.environment = environment
        self.gamma = gamma
        self.alpha = alpha
        self.epsilon = epsilon
        self.episodes = episodes
        self.Q_table = dict() # Initialize Q-table to store state-action values
        # Initialize the value of each state-action pair to 0
        #Initialize the Q_table as map of maps:
        #- each state maps to a map of actions
        #-each action maps to a value
        for state in environment.get_states():
            self.Q_table[state] = dict()
            for action in environment.get_actions(state):
                self.Q_table[state][action] = 0
        # Use epsilon-greedy policy for learning
        self.policy = EpsilonGreedyPolicy(environment, self.Q_table, epsilon)
        
    # Apply the SARSA algorithm to learn and update the Q-values       
    def apply(self):
        
        for e in range(self.episodes):
            self.episode_length=0
            state = self.environment.current_state 
            action = self.policy.apply(self.environment.current_state) 
            while not self.environment.task_done() :
                reward = self.environment.do_action_and_get_reward(action)
                next_state = self.environment.current_state
                next_action = self.policy.apply(next_state)
                temp_diff = self.Q_table[next_state][next_action] - self.Q_table[state][action]
                self.Q_table[state][action] =\
                self.Q_table[state][action] + self.alpha * (reward + self.gamma * temp_diff) 
                state = next_state
                action = next_action
                self.episode_length +=1
            
            # Must reset the environment before trying another episode
            self.environment.reset()
            print(f'Episode {e+1},Length: {self.episode_length}')

## Experimenting the SARSA algorithm

In [ ]:
Walking_Cliff = WalkingCliff()
# gamma 0.9
# alpha = 0.2
# epsilon = 0.1 
# Is there a better choice? Maybe, need to experiment with grid search
sarsa = SARSA(Walking_Cliff, 0.9, 0.2, 0.1, 1000)

In [ ]:

sarsa.apply()


The GreedyPolicy class implements a policy that always selects the action with the highest estimated value according to the Q-table.

To avoid reading the Q_table directly, I implement a greedy policy class based on Q_table values: Printing the optimal policy  ??? based on the  ??? values approximated by SARSA. Creating a new class for a greedy policy:
  
  

In [ ]:

class GreedyPolicy(BasePolicy):
    
    def __init__(self, environment, Q_table):
        self.environment = environment
        self.Q_table = Q_table
        
    def apply(self, state):
        actions = self.environment.get_actions(state)
        max_action_value = -1 * sys.float_info.max
        best_action = None
        for action in actions:
            if self.Q_table[state][action] > max_action_value:
                max_action_value = self.Q_table[state][action]
                best_action = action
        return best_action

In [ ]:
greedy_policy = GreedyPolicy(Walking_Cliff, sarsa.Q_table)

As we can see below there is the optimal policy based on q-table

In [ ]:
for ri in range(4):
    for co in range(12):
        print(WalkingCliff.label[greedy_policy.apply((ri,co))],end=' ')
    print()

Printing the values in the  ??
 -table:

In [ ]:
for ri in range(4):
    for co in range(12):
        print("({},{}):".format(ri,co))
        actions = sarsa.environment.get_actions((ri,co))
        for action in actions:
            print("{} -> {}".format(WalkingCliff.label[action],sarsa.Q_table[(ri,co)][action]))
        print

Above there are the values of the Q_table where are the weighted sum of the all reward 

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from IPython.display import display, clear_output
import time
class WalkingCliffVisualizer:
    def __init__(self, Walking_Cliff, q_table):
        self.Walking_Cliff = Walking_Cliff
        self.q_table = q_table
        self.fig, self.ax = plt.subplots()

    def plot_grid(self, agent_position):
        self.ax.clear()
        for ri in range(4):
            for co in range(12):
                if co in range(1,11) and ri==3:
                    color = 'red'
                elif (ri, co) == (3,11):
                    color = 'green'
                elif (ri, co) == (3,0):
                    color = 'yellow'
                else:
                    color = 'white'

                rect = patches.Rectangle((co, ri), 1, 1, linewidth=1, edgecolor='black', facecolor=color)
                self.ax.add_patch(rect)
                
        self.ax.text(agent_position[1] + 0.5, agent_position[0] + 0.5, 'Agent', ha='center', va='center', color='blue')

        self.ax.set_xlim(0, 12)
        self.ax.set_ylim(0, 4)
        self.ax.invert_yaxis() 
        display(self.fig)
        clear_output(wait=True)
        time.sleep(0.00003)  

    def animate_agent(self, num_episodes):
        for episode in range(num_episodes):
            self.Walking_Cliff.reset()
            self.Walking_Cliff.current_state = (3, 0)  
            self.Walking_Cliff.get_states()
            
            while not self.Walking_Cliff.task_done():
                state = self.Walking_Cliff.current_state
                #action = GreedyPolicy(self.Walking_Cliff, self.q_table).apply(state)
                action = sarsa.policy.apply(state)
                self.Walking_Cliff.do_action_and_get_reward(action)
                self.plot_grid(self.Walking_Cliff.current_state)
                time.sleep(0.00003)  # Ritardo di 0.3 secondi tra le azioni

        # Mostra la tabella Q alla fine di ogni episodio
        print("Q-table values at the end of episode", episode+1)
        for state, actions in sarsa.Q_table.items():
            print("State:", state)
            for action, value in actions.items():
                print("Action:", WalkingCliff.label[action], "Value:", value)
            print()

        #env.close()

### Run the code below to see the exploiaration of the agent in the gridword under the rule of walking cliff

In [ ]:
visualizer = WalkingCliffVisualizer(Walking_Cliff, sarsa.Q_table)
visualizer.animate_agent(num_episodes=1)